# Cinegraph - PySpark / SparkSQL desde S3

In [ ]:
!pip -q install pyspark==4.1.2 boto3

In [ ]:
from google.colab import userdata

AWS_KEY = userdata.get("aws_access_key_id").strip()
AWS_SECRET = userdata.get("aws_secret_access_key").strip()
AWS_TOKEN = userdata.get("aws_session_token").strip()
AWS_REGION = userdata.get("aws_default_region").strip()

BUCKET = "movie-analytics-lake"

BOTO_KW = {
    "aws_access_key_id": AWS_KEY,
    "aws_secret_access_key": AWS_SECRET,
    "aws_session_token": AWS_TOKEN,
    "region_name": AWS_REGION,
}

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("cinegraph-sparksql")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
import boto3
from pathlib import Path

LOCAL = Path("/content/trusted")
s3 = boto3.client("s3", **BOTO_KW)

for tabla in ["movies", "movie_genres", "people", "movie_reviews", "exchange"]:
    prefijo = f"trusted/{tabla}/"
    dest = LOCAL / tabla
    dest.mkdir(parents=True, exist_ok=True)
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefijo):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith(".parquet"):
                continue
            s3.download_file(BUCKET, key, str(dest / key.split("/")[-1]))
    print(tabla, "ok")

RUTA = str(LOCAL) + "/"
spark.read.parquet(RUTA + "movies/").createOrReplaceTempView("movies")
spark.read.parquet(RUTA + "movie_genres/").createOrReplaceTempView("movie_genres")
spark.read.parquet(RUTA + "people/").createOrReplaceTempView("people")
spark.read.parquet(RUTA + "movie_reviews/").createOrReplaceTempView("movie_reviews")
spark.read.parquet(RUTA + "exchange/").createOrReplaceTempView("exchange")

print("peliculas:", spark.sql("SELECT COUNT(*) n FROM movies").collect()[0].n)

## Pregunta 1

In [ ]:
spark.sql("""
SELECT
    g.genre,
    COUNT(DISTINCT g.tmdb_id)              AS num_peliculas,
    ROUND(AVG(m.revenue_usd), 2)           AS avg_revenue_usd,
    ROUND(SUM(m.revenue_usd), 2)           AS total_revenue_usd,
    ROUND(AVG(m.revenue_cop), 2)           AS avg_revenue_cop,
    ROUND(SUM(m.revenue_cop), 2)           AS total_revenue_cop
FROM movie_genres g
JOIN movies m ON g.tmdb_id = m.tmdb_id
WHERE m.revenue_usd IS NOT NULL
  AND m.revenue_cop IS NOT NULL
GROUP BY g.genre
ORDER BY total_revenue_usd DESC
LIMIT 20
""").show(truncate=False)

## Pregunta 2

In [ ]:
spark.sql("""
WITH rentables AS (
    SELECT tmdb_id, profit_usd, cast_ids
    FROM movies
    WHERE profit_usd IS NOT NULL
      AND profit_usd > 0
),
cast_expandido AS (
    SELECT
        r.tmdb_id,
        r.profit_usd,
        TRIM(actor_id_raw) AS actor_id_str
    FROM rentables r
    LATERAL VIEW explode(split(r.cast_ids, ',')) t AS actor_id_raw
    WHERE r.cast_ids IS NOT NULL
      AND TRIM(actor_id_raw) <> ''
)
SELECT
    p.tmdb_id,
    p.name                                 AS actor,
    COUNT(DISTINCT c.tmdb_id)              AS peliculas_rentables,
    ROUND(AVG(c.profit_usd), 2)            AS avg_profit_usd
FROM cast_expandido c
JOIN people p
  ON CAST(c.actor_id_str AS INTEGER) = p.tmdb_id
GROUP BY p.tmdb_id, p.name
ORDER BY peliculas_rentables DESC, avg_profit_usd DESC
LIMIT 25
""").show(truncate=False)

## Pregunta 3

In [ ]:
spark.sql("""
SELECT
    release_year,
    COUNT(*)                               AS peliculas,
    ROUND(SUM(revenue_usd), 2)             AS total_revenue_usd,
    ROUND(SUM(revenue_cop), 2)             AS total_revenue_cop,
    ROUND(AVG(revenue_cop / NULLIF(revenue_usd, 0)), 4) AS tasa_cop_aplicada
FROM movies
WHERE release_year IS NOT NULL
  AND revenue_usd IS NOT NULL
  AND revenue_cop IS NOT NULL
GROUP BY release_year
HAVING COUNT(*) >= 5
ORDER BY release_year
""").show(truncate=False)

## Pregunta 4

In [ ]:
spark.sql("""
WITH fx AS (
    SELECT rate AS cop_rate
    FROM exchange
    WHERE currency = 'COP'
    LIMIT 1
),
por_director AS (
    SELECT
        TRIM(director_name)                AS director,
        m.tmdb_id,
        m.roi_pct,
        m.profit_usd,
        m.revenue_usd,
        m.budget_usd,
        m.revenue_usd * fx.cop_rate        AS revenue_cop_calc,
        m.budget_usd * fx.cop_rate         AS budget_cop,
        CASE
            WHEN m.budget_usd IS NOT NULL AND m.budget_usd > 0
            THEN (
                (m.revenue_usd * fx.cop_rate) - (m.budget_usd * fx.cop_rate)
            ) / (m.budget_usd * fx.cop_rate) * 100
        END                                AS roi_adjusted
    FROM movies m
    CROSS JOIN fx
    LATERAL VIEW explode(split(m.directors, ',')) t AS director_name
    WHERE m.directors IS NOT NULL
      AND TRIM(director_name) <> ''
)
SELECT
    director,
    COUNT(DISTINCT tmdb_id)                AS peliculas,
    ROUND(AVG(roi_pct), 2)                 AS avg_roi_pct,
    ROUND(AVG(roi_adjusted), 2)            AS avg_roi_adjusted_cop,
    ROUND(AVG(profit_usd), 2)              AS avg_profit_usd,
    ROUND(SUM(revenue_usd), 2)             AS total_revenue_usd
FROM por_director
WHERE roi_pct IS NOT NULL
GROUP BY director
HAVING COUNT(DISTINCT tmdb_id) >= 3
ORDER BY avg_roi_pct DESC
LIMIT 20
""").show(truncate=False)

## Pregunta 5

In [ ]:
spark.sql("""
WITH review_agg AS (
    SELECT
        tmdb_id,
        AVG(rating)                          AS avg_review_rating
    FROM movie_reviews
    WHERE rating IS NOT NULL
    GROUP BY tmdb_id
    HAVING COUNT(*) >= 3
)
SELECT
    CORR(m.vote_average, m.revenue_usd)      AS corr_nota_tmdb_vs_revenue,
    CORR(m.vote_average, m.profit_usd)       AS corr_nota_tmdb_vs_profit,
    CORR(m.vote_average, m.roi_pct)          AS corr_nota_tmdb_vs_roi,
    CORR(r.avg_review_rating, m.revenue_usd) AS corr_resenas_vs_revenue,
    CORR(r.avg_review_rating, m.profit_usd)  AS corr_resenas_vs_profit,
    CORR(r.avg_review_rating, m.roi_pct)     AS corr_resenas_vs_roi
FROM movies m
JOIN review_agg r ON m.tmdb_id = r.tmdb_id
WHERE m.vote_average IS NOT NULL
  AND m.revenue_usd IS NOT NULL
""").show(truncate=False)

## KPIs

In [ ]:
spark.sql("""
SELECT COUNT(*) AS total_peliculas,
       ROUND(SUM(revenue_usd), 2) AS revenue_total_usd,
       ROUND(SUM(revenue_cop), 2) AS revenue_total_cop,
       ROUND(AVG(roi_pct), 2) AS roi_promedio
FROM movies
""").show()

In [ ]:
spark.stop()